# Matched chest-lead masking control

This sequential follow-up to `Lead_Availability_Probe.ipynb` masks V1/V3/V5 on the identical twelve-lead validation records. It compares that intervention to masking V2/V4/V6 with both frozen models and the same preprocessing. It is a control for *number* and broad location of removed leads, not a fully equivalent signal removal: the two groups contain different information. No training or test-set access occurs. Run the first notebook before this one to produce `paired_predictions.csv`.

In [1]:
from pathlib import Path
import csv, importlib.util, json, os, random, sys
import numpy as np, pandas as pd, torch, wfdb
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Data/AttributesDictionary.csv').is_file())
OUT=ROOT/'new work/Week5/experimental/lead_availability_probe'
RECORDS=ROOT/'Child_ECG/Child_ecg'; FS=500; N_SAMPLES=5000; SEED=42
assert Path(sys.executable).resolve()==(ROOT/'venv/bin/python').resolve()
assert torch.cuda.is_available(), 'This GPU experiment requires CUDA in the notebook kernel.'
DEVICE=torch.device('cuda'); torch.set_num_threads(min(4,os.cpu_count() or 1))
torch.manual_seed(SEED); np.random.seed(SEED)
print('Python',sys.executable,'PyTorch',torch.__version__,'CUDA',torch.cuda.get_device_name(0))

Python /home/souhit/Documents/Research/Pediatric_ECG/venv/bin/python PyTorch 2.6.0+cu124 CUDA NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
preds=pd.read_csv(ROOT/'new work/Week5/experimental/validation_predictions.csv')
val_ids=set(preds.patient_id.astype(str))
selected_codes=set(pd.read_csv(ROOT/'Data/DiseaseCode.csv').query('`Disease Type` != "Other diseases(OD)"')['ICD-10 Code'].astype(str).str.strip())
assert len(selected_codes)==19
def tokens(s): return [v.strip().strip("'\"").strip() for v in s.split(';') if v.strip()]
rows=[]
with (ROOT/'Data/AttributesDictionary.csv').open(newline='') as f:
    for row in csv.DictReader(f):
        if row['Patient_ID'] not in val_ids: continue
        rows.append({'patient_id':row['Patient_ID'],'Filename':row['Filename'],'Sampling_point':int(row['Sampling_point']),
                     'Lead':int(row['Lead']),'target':int(bool(set(tokens(row['ICD-10 code']))&selected_codes))})
val=pd.DataFrame(rows)
assert len(val)==len(preds)==2169
assert (val.patient_id.to_numpy()==preds.patient_id.astype(str).to_numpy()).all()
assert (val.target.to_numpy()==preds.target.to_numpy()).all()
subset=val[val.Lead==12].copy(); subset['validation_index']=subset.index
assert len(subset)==1888 and subset.target.sum()==426
print('12-lead validation records',len(subset),'positives',int(subset.target.sum()),'prevalence',subset.target.mean())

12-lead validation records 1888 positives 426 prevalence 0.225635593220339


In [3]:
LEADS=['I','II','III','AVR','AVL','AVF','V1','V2','V3','V4','V5','V6']; lead_index={v:i for i,v in enumerate(LEADS)}
MASKED_LEADS=['V2','V4','V6']; masked_indices=[lead_index[v] for v in MASKED_LEADS]
class ECGValidation(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        sig,fields=wfdb.rdsamp(str(RECORDS/r.Filename),sampto=min(N_SAMPLES,int(r.Sampling_point)))
        assert fields['fs']==FS and np.isfinite(sig).all() and len(fields['sig_name'])==12
        x=np.zeros((12,N_SAMPLES),dtype=np.float32)
        for j,lead in enumerate(fields['sig_name']):
            k=lead_index[lead.upper()]; v=sig[:,j].astype(np.float32); sd=v.std()
            x[k,:len(v)]=(v-v.mean())/sd if sd>1e-8 else 0
        return torch.from_numpy(x),int(r.target)
loader=DataLoader(ECGValidation(subset),batch_size=16,shuffle=False,num_workers=2)
def import_class(name,path,class_name):
    spec=importlib.util.spec_from_file_location(name,path); module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module); return getattr(module,class_name)
ResNet1D=import_class('week5_models',ROOT/'NoteBooks/Week5/models.py','ResNet1D')
MultiScaleResNet1D=import_class('week8_models',ROOT/'NoteBooks/Week8/ExperimentB/models.py','MultiScaleResNet1D')
model_specs={'week5_resnet_unweighted':(ResNet1D,ROOT/'NoteBooks/Week5/disease_labelled/resnet_unweighted_best.pth'),
             'multiscale_resnet':(MultiScaleResNet1D,ROOT/'new work/Week5/experimental/multiscale_resnet_best.pth')}
def load_model(cls,path):
    ckpt=torch.load(path,map_location=DEVICE,weights_only=False)
    assert ckpt['lead_order']==LEADS and ckpt['input_samples']==N_SAMPLES
    net=cls(2).to(DEVICE);net.load_state_dict(ckpt['model_state_dict']);net.eval()
    return net
models={name:load_model(*spec) for name,spec in model_specs.items()}
print('Frozen checkpoints loaded:',list(models))

Frozen checkpoints loaded: ['week5_resnet_unweighted', 'multiscale_resnet']


In [4]:
prior=pd.read_csv(OUT/'paired_predictions.csv')
assert np.array_equal(prior.validation_index.to_numpy(),subset.validation_index.to_numpy())
assert np.array_equal(prior.target.to_numpy(),subset.target.to_numpy())
CONTROL_LEADS=['V1','V3','V5']; control_indices=[lead_index[v] for v in CONTROL_LEADS]
control={name:[] for name in models};ys=[]
with torch.inference_mode():
    for batch,(x,y) in enumerate(loader):
        x=x.to(DEVICE);x[:,control_indices,:]=0
        ys.extend(y.tolist())
        for name,model in models.items():
            control[name].extend(torch.softmax(model(x),dim=1)[:,1].cpu().numpy().tolist())
        if (batch+1)%25==0:print('processed',min((batch+1)*16,len(subset)),'/',len(subset),flush=True)
assert np.array_equal(np.asarray(ys),prior.target.to_numpy())
for name in models:prior[name+'_control']=control[name]
prior.to_csv(OUT/'control_predictions.csv',index=False)
print('Saved matched control predictions')

processed 400 / 1888


processed 800 / 1888


processed 1200 / 1888


processed 1600 / 1888


Saved matched control predictions


In [5]:
def score(y,p):
    y=np.asarray(y,dtype=int);p=np.asarray(p);yp=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    return {'n':len(y),'positive_n':int(y.sum()),'prevalence':y.mean(),
            'accuracy':accuracy_score(y,yp),'sensitivity':tp/(tp+fn),'specificity':tn/(tn+fp),
            'precision':precision_score(y,yp,zero_division=0),'f1':f1_score(y,yp,zero_division=0),
            'auroc':roc_auc_score(y,p),'average_precision':average_precision_score(y,p),
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}
rows=[]
for name in models:
    for condition in ['original','masked','control']:
        rows.append({'model':name,'condition':condition,**score(prior.target,prior[name+'_'+condition])})
metrics=pd.DataFrame(rows);metrics.to_csv(OUT/'control_metrics.csv',index=False)
display(metrics.round(4))
# Compare each masking effect to the original, then the two masking effects to each other.
comparison=[]
for name in models:
    m={condition:score(prior.target,prior[name+'_'+condition]) for condition in ['original','masked','control']}
    for condition in ['masked','control']:
        comparison.append({'model':name,'condition':condition,
                           **{k+'_change':m[condition][k]-m['original'][k] for k in ['accuracy','sensitivity','specificity','f1','auroc','average_precision']},
                           'mean_absolute_probability_change':float((prior[name+'_'+condition]-prior[name+'_original']).abs().mean()),
                           'positive_to_negative':int(((prior[name+'_original']>=.5)&(prior[name+'_'+condition]<.5)).sum()),
                           'negative_to_positive':int(((prior[name+'_original']<.5)&(prior[name+'_'+condition]>=.5)).sum())})
comparison=pd.DataFrame(comparison);comparison.to_csv(OUT/'control_comparison.csv',index=False)
display(comparison.round(4))

,model,condition,n,positive_n,prevalence,accuracy,sensitivity,specificity,precision,f1,auroc,average_precision,tn,fp,fn,tp
0,week5_resnet_unweighted,original,1888,426,0.2256,0.8268,0.3427,0.9679,0.7565,0.4717,0.7869,0.6076,1415,47,280,146
1,week5_resnet_unweighted,masked,1888,426,0.2256,0.7082,0.5610,0.7510,0.3964,0.4645,0.7208,0.5137,1098,364,187,239
2,week5_resnet_unweighted,control,1888,426,0.2256,0.8072,0.2700,0.9637,0.6845,0.3872,0.7532,0.5365,1409,53,311,115
3,multiscale_resnet,original,1888,426,0.2256,0.8279,0.3944,0.9542,0.7149,0.5083,0.7946,0.6237,1395,67,258,168
4,multiscale_resnet,masked,1888,426,0.2256,0.7458,0.5822,0.7934,0.4509,0.5082,0.7473,0.5345,1160,302,178,248
5,multiscale_resnet,control,1888,426,0.2256,0.7924,0.4624,0.8885,0.5472,0.5013,0.7633,0.5413,1299,163,229,197


,model,condition,accuracy_change,sensitivity_change,specificity_change,f1_change,auroc_change,average_precision_change,mean_absolute_probability_change,positive_to_negative,negative_to_positive
0,week5_resnet_unweighted,masked,-0.1186,0.2183,-0.2168,-0.0072,-0.0660,-0.0939,0.2204,3,413
1,week5_resnet_unweighted,control,-0.0196,-0.0728,-0.0041,-0.0845,-0.0337,-0.0711,0.0824,68,43
2,multiscale_resnet,masked,-0.0821,0.1878,-0.1607,-0.0001,-0.0473,-0.0892,0.1973,5,320
3,multiscale_resnet,control,-0.0355,0.0681,-0.0657,-0.0070,-0.0313,-0.0824,0.1159,33,158


In [6]:
# Paired patient bootstrap of the difference in masking effects: optional-lead mask minus control mask.
keys=np.unique(prior.patient_id.to_numpy());ix_by_patient={k:np.flatnonzero(prior.patient_id.to_numpy()==k) for k in keys}
rng=np.random.default_rng(SEED);boot=[]
for rep in range(300):
    ix=np.concatenate([ix_by_patient[k] for k in rng.choice(keys,size=len(keys),replace=True)])
    y=prior.target.to_numpy()[ix]
    if len(np.unique(y))!=2:continue
    for name in models:
        optional=score(y,prior[name+'_masked'].to_numpy()[ix]);control_score=score(y,prior[name+'_control'].to_numpy()[ix])
        boot.append({'replicate':rep,'model':name,**{k+'_optional_minus_control':optional[k]-control_score[k] for k in ['sensitivity','specificity','f1','auroc','average_precision']}})
boot=pd.DataFrame(boot)
summary=[]
for name,frame in boot.groupby('model'):
    for metric in ['sensitivity','specificity','f1','auroc','average_precision']:
        values=frame[metric+'_optional_minus_control']
        summary.append({'model':name,'metric':metric,'mean':values.mean(),'q025':values.quantile(.025),'q975':values.quantile(.975)})
summary=pd.DataFrame(summary);summary.to_csv(OUT/'control_patient_bootstrap.csv',index=False)
display(summary.round(4))

,model,metric,mean,q025,q975
0,multiscale_resnet,sensitivity,0.1217,0.0789,0.1633
1,multiscale_resnet,specificity,-0.0945,-0.1133,-0.0744
2,multiscale_resnet,f1,0.0089,-0.0264,0.0449
3,multiscale_resnet,auroc,-0.0152,-0.0352,0.0079
4,multiscale_resnet,average_precision,-0.0054,-0.0461,0.0331
5,week5_resnet_unweighted,sensitivity,0.2930,0.2333,0.3603
6,week5_resnet_unweighted,specificity,-0.2134,-0.2370,-0.1926
7,week5_resnet_unweighted,f1,0.0788,0.0262,0.1432
8,week5_resnet_unweighted,auroc,-0.0324,-0.0661,0.0004
9,week5_resnet_unweighted,average_precision,-0.0223,-0.0725,0.0298


## Interpretation and limits

**Same-record comparison:** 1,888 twelve-lead validation records from 1,555 patients, including 426 selected-code positives (22.56%). Both frozen models lost ranking performance when either three-lead group was zeroed. Relative to the unmasked input, the multi-scale ResNet's AUROC/AP changed by −0.0473/−0.0892 for V2/V4/V6 and −0.0313/−0.0824 for V1/V3/V5. The Week 5 unweighted ResNet's AUROC/AP changed by −0.0660/−0.0939 and −0.0337/−0.0711, respectively. Thus neither intervention improves discrimination.

**Asymmetric threshold response:** For the multi-scale ResNet, V2/V4/V6 masking raised sensitivity from 0.3944 to 0.5822 and lowered specificity from 0.9542 to 0.7934. The V1/V3/V5 control raised sensitivity only to 0.4624 and lowered specificity to 0.8885. For the Week 5 ResNet, the optional-lead mask raised sensitivity from 0.3427 to 0.5610 and lowered specificity from 0.9679 to 0.7510; the control mask *lowered* sensitivity to 0.2700 while leaving specificity at 0.9637. The first intervention causes a larger positive-score shift, especially for the original ResNet. The paired patient bootstrap estimates of optional-minus-control sensitivity were +0.122 (descriptive 95% interval +0.079 to +0.163) for multi-scale and +0.293 (+0.233 to +0.360) for the original ResNet. The corresponding specificity differences were −0.094 (−0.113 to −0.074) and −0.213 (−0.237 to −0.193). AP and AUROC differences between the two masks had intervals including zero for the multi-scale model.

**Interpretation:** The two models respond differently to which chest leads are removed. This supports an *asymmetric dependence on lead content or configuration* in this validation subset. It does not identify a lead-presence shortcut: the masked sets contain different ECG information, and zeroing real leads creates a distribution shift unlike naturally recorded nine-lead ECGs. All intervals are descriptive because the same validation set was used in previous model selection. No test set, training, threshold tuning, or clinical causality claim is involved.